
# Aula Prática — Chatbot com NLP e Aprendizado de Máquina

**Curso:** Ciência da Computação - Uniderp

**Tema:** Aprendizado de Máquina + NLU/NLG  

**Ambiente:** Jupyter Notebook / JupyterLab / Google Colab

**Professor:** Murilo Gustavo Nabarrete Costa

## Objetivo
Construir, passo a passo, um chatbot simples para atendimento de uma loja virtual.




## Arquitetura que vamos implementar

```text
Mensagem do usuário
        ↓
Pré-processamento
        ↓
Representação numérica do texto
        ↓
Classificador de intenção
        ↓
Intenção + entidade
        ↓
Gerador de resposta
        ↓
Resposta do chatbot
```

### Intenções

Nosso chatbot terá quatro intenções:

- `saudacao`
- `preco`
- `estoque`
- `pedido`

Exemplos:

| Mensagem | Intenção |
|---|---|
| "Oi" | saudacao |
| "Quanto custa o notebook?" | preco |
| "Tem o celular em estoque?" | estoque |
| "Quero saber onde está meu pedido" | pedido |

A ideia central é que **mensagens diferentes podem representar a mesma intenção**.


In [1]:
#Preparação do ambiente Local

# Se alguma biblioteca não estiver instalada, execute esta célula.
%pip install -q pandas scikit-learn matplotlib


In [2]:

import re
import random
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report



## Criando os dados de treinamento

Em **aprendizado supervisionado**, o modelo recebe exemplos de entrada associados a uma saída conhecida.

Aqui:

- entrada = mensagem do usuário;
- saída = intenção correta.

Esse conjunto pequeno será nosso **dataset de treinamento**.

> Em um sistema real, precisaríamos de muito mais exemplos e de uma estratégia cuidadosa para construção e validação do dataset.


In [35]:

dados = [
    # saudacao
    ("oi", "saudacao"),
    ("olá", "saudacao"),
    ("bom dia", "saudacao"),
    ("boa tarde", "saudacao"),
    ("boa noite", "saudacao"),
    ("oi tudo bem", "saudacao"),
    ("olá gostaria de ajuda", "saudacao"),
    ("e aí beleza", "saudacao"),
    ("oii", "saudacao"),
    ("bom dia tudo certo", "saudacao"),
    ("alguém pode me ajudar", "saudacao"),
    ("preciso de uma informação", "saudacao"),

    # preco
    ("qual o preço do notebook", "preco"),
    ("quanto custa o celular", "preco"),
    ("qual o valor desse produto", "preco"),
    ("quero saber o preço do computador", "preco"),
    ("me informe o valor do smartphone", "preco"),
    ("quanto vou pagar pelo notebook", "preco"),
    ("esse produto custa quanto", "preco"),
    ("me diga quanto custa esse computador", "preco"),
    ("qual o valor do tablet", "preco"),
    ("está com desconto o notebook", "preco"),
    ("valor do fone de ouvido", "preco"),
    ("quanto sai esse celular", "preco"),

    # estoque
    ("tem notebook em estoque", "estoque"),
    ("o celular está disponível", "estoque"),
    ("vocês têm esse produto", "estoque"),
    ("tem o smartphone disponível", "estoque"),
    ("quero saber se o computador está em estoque", "estoque"),
    ("esse produto ainda está disponível", "estoque"),
    ("posso comprar esse produto agora", "estoque"),
    ("tem esse notebook para pronta entrega", "estoque"),
    ("ainda tem unidade desse produto", "estoque"),
    ("o tablet está em estoque", "estoque"),
    ("consigo comprar hoje esse celular", "estoque"),
    ("vocês têm essa cor disponível", "estoque"),

    # pedido
    ("onde está meu pedido", "pedido"),
    ("quero acompanhar meu pedido", "pedido"),
    ("qual o status da minha entrega", "pedido"),
    ("meu pedido já foi enviado", "pedido"),
    ("quando meu pedido vai chegar", "pedido"),
    ("quero rastrear minha compra", "pedido"),
    ("como acompanho a entrega", "pedido"),
    ("quero saber onde está minha compra", "pedido"),
    ("meu pacote já saiu para entrega", "pedido"),
    ("cadê meu pedido", "pedido"),
    ("qual o prazo de entrega do meu pedido", "pedido"),
    ("preciso rastrear minha encomenda", "pedido"),

    # devolucao
    ("quero devolver meu produto", "devolucao"),
    ("como faço uma devolução", "devolucao"),
    ("preciso devolver minha compra", "devolucao"),
    ("quero solicitar devolução", "devolucao"),
    ("como funciona a devolução", "devolucao"),
    ("quero trocar esse produto", "devolucao"),
    ("posso devolver um item com defeito", "devolucao"),
    ("como faço para estornar minha compra", "devolucao"),
    ("preciso cancelar e devolver o pedido", "devolucao"),
    ("quero reembolso desse produto", "devolucao"),
    ("o produto veio errado, quero devolver", "devolucao"),
    ("qual o prazo para devolução", "devolucao"),
    ("meu produto chegou quebrado, quero devolver", "devolucao"),
    ("como funciona a política de troca", "devolucao"),
]

df = pd.DataFrame(dados, columns=["texto", "intencao"])
df


,texto,intencao
0,oi,saudacao
1,olá,saudacao
2,bom dia,saudacao
3,boa tarde,saudacao
4,boa noite,saudacao
...,...,...
57,quero reembolso desse produto,devolucao
58,"o produto veio errado, quero devolver",devolucao
59,qual o prazo para devolução,devolucao
60,"meu produto chegou quebrado, quero devolver",devolucao



## Conhecendo os dados

Antes de treinar um modelo, observe os exemplos.

**Discussão:** essa abordagem baseada em regras pode funcionar para exemplos simples, mas tende a falhar quando a mesma intenção é expressa de várias maneiras.

É justamente aí que entra a classificação de intenções.


In [36]:

print(df["intencao"].value_counts())


intencao
devolucao    14
saudacao     12
preco        12
estoque      12
pedido       12
Name: count, dtype: int64



## Separando treinamento e teste

Vamos separar os exemplos em dois grupos:

- **treinamento:** exemplos utilizados pelo modelo para aprender;
- **teste:** exemplos reservados para verificar como o modelo se comporta diante de dados que não foram utilizados no treinamento.

Como o dataset é pequeno, esta divisão é apenas didática.


In [37]:

X_train, X_test, y_train, y_test = train_test_split(
    df["texto"],
    df["intencao"],
    test_size=0.25,
    random_state=42,
    stratify=df["intencao"]
)

print("Treinamento:", len(X_train))
print("Teste:", len(X_test))


Treinamento: 46
Teste: 16



## Transformando texto em números

Computadores e algoritmos de ML não trabalham diretamente com o significado humano das frases. Precisamos transformar o texto em uma representação numérica.

Vamos utilizar:

**CountVectorizer → LogisticRegression**

O `CountVectorizer` transforma as palavras em características numéricas com base em sua ocorrência nos textos.

A `LogisticRegression` será utilizada como classificador das intenções.


In [38]:

modelo = Pipeline([
    ("vetorizador", CountVectorizer()),
    ("classificador", LogisticRegression(max_iter=1000))
])

modelo.fit(X_train, y_train)

print("Modelo treinado!")


Modelo treinado!



## Testando o classificador

Agora vamos fornecer mensagens que o modelo não recebeu exatamente dessa forma durante o treinamento.

Observe principalmente a **intenção prevista**.


In [39]:

mensagens = [
    "quanto custa o smartphone",
    "vocês têm notebook disponível?",
    "quero acompanhar a minha entrega",
    "olá, preciso de ajuda",
    "preciso devolver minha compra"
]

previsoes = modelo.predict(mensagens)

for mensagem, previsao in zip(mensagens, previsoes):
    print(f"Mensagem: {mensagem}")
    print(f"Intenção: {previsao}")
    print("-" * 50)


Mensagem: quanto custa o smartphone
Intenção: preco
--------------------------------------------------
Mensagem: vocês têm notebook disponível?
Intenção: estoque
--------------------------------------------------
Mensagem: quero acompanhar a minha entrega
Intenção: pedido
--------------------------------------------------
Mensagem: olá, preciso de ajuda
Intenção: saudacao
--------------------------------------------------
Mensagem: preciso devolver minha compra
Intenção: devolucao
--------------------------------------------------


### Teste de processo completo detalhado

In [40]:
mensagem = "quanto custa o notebook?"


vetorizador = modelo.named_steps["vetorizador"]

print("# 1. Transformar texto em números: \n")
print( vetorizador.vocabulary_, '\n')

X = vetorizador.transform([mensagem])

print("# 2. Vetor de características: \n")
print(X.toarray(), "\n")

classificador = modelo.named_steps["classificador"]

print("# 3. Probabilidades: \n")
print(classificador.predict_proba(X)[0], "\n")

probabilidades = classificador.predict_proba(X)[0]

print("Mensagem:", mensagem, "\n")

for classe, probabilidade in zip(
    classificador.classes_,
    probabilidades
):
    print(f"{classe:10} → {probabilidade:.2%}")

print("\n Intenção prevista:", classificador.classes_[probabilidades.argmax()])

# 1. Transformar texto em números: 

{'oi': 54, 'alguém': 6, 'pode': 62, 'me': 50, 'ajudar': 5, 'qual': 69, 'prazo': 65, 'para': 59, 'devolução': 29, 'quero': 73, 'saber': 76, 'preço': 67, 'do': 34, 'computador': 21, 'olá': 55, 'gostaria': 46, 'de': 26, 'ajuda': 4, 'trocar': 85, 'esse': 39, 'produto': 68, 'meu': 51, 'pedido': 60, 'já': 49, 'foi': 43, 'enviado': 37, 'vocês': 92, 'têm': 87, 'ainda': 3, 'tem': 83, 'unidade': 89, 'desse': 28, 'boa': 10, 'tarde': 82, 'aí': 7, 'beleza': 8, 'status': 80, 'da': 25, 'minha': 52, 'entrega': 36, 'preciso': 66, 'cancelar': 13, 'devolver': 30, 'cadê': 12, 'está': 41, 'disponível': 33, 'acompanhar': 0, 'compra': 19, 'quando': 70, 'vai': 90, 'chegar': 15, 'valor': 91, 'quanto': 71, 'sai': 77, 'celular': 14, 'tablet': 81, 'em': 35, 'estoque': 40, 'diga': 32, 'custa': 24, 'rastrear': 74, 'uma': 88, 'informação': 48, 'onde': 56, 'com': 17, 'desconto': 27, 'notebook': 53, 'como': 18, 'funciona': 45, 'política': 63, 'troca': 84, 'essa': 38, 'cor': 23, 'fo

### O modelo aprende pesos associados às características. Esses pesos contribuem para decidir qual classe é mais provável.

In [41]:
features = vetorizador.get_feature_names_out()

pesos_df = pd.DataFrame(
    classificador.coef_,
    columns=features,
    index=classificador.classes_
)

pesos_df

,acompanhar,acompanho,agora,ainda,ajuda,ajudar,alguém,aí,beleza,bem,...,troca,trocar,tudo,têm,uma,unidade,vai,valor,vocês,vou
devolucao,-0.131736,-0.150661,-0.099053,-0.158057,-0.064090,-0.072556,-0.072556,-0.100115,-0.100115,-0.068674,...,0.373948,0.383328,-0.068674,-0.155777,0.131470,-0.102705,-0.059501,-0.243622,-0.155777,-0.045230
estoque,-0.024286,-0.081739,0.255160,0.442843,-0.048471,-0.067235,-0.067235,-0.094152,-0.094152,-0.064945,...,-0.059126,-0.167395,-0.064945,0.513758,-0.111426,0.276563,-0.039814,-0.177137,0.513758,-0.048335
pedido,0.234371,0.453769,-0.029664,-0.055040,-0.049537,-0.062644,-0.062644,-0.084823,-0.084823,-0.059036,...,-0.089588,-0.063682,-0.059036,-0.078154,-0.120601,-0.032001,0.205918,-0.115153,-0.078154,-0.039025
preco,-0.033525,-0.079718,-0.065235,-0.131325,-0.062306,-0.086280,-0.086280,-0.099826,-0.099826,-0.068486,...,-0.074943,-0.092544,-0.068486,-0.123287,-0.126263,-0.073760,-0.038468,0.753096,-0.123287,0.208555
saudacao,-0.044825,-0.141651,-0.061208,-0.098422,0.224403,0.288716,0.288716,0.378916,0.378916,0.261141,...,-0.150292,-0.059706,0.261141,-0.156540,0.226820,-0.068097,-0.068135,-0.217184,-0.156540,-0.075966



## Avaliando o modelo
Vamos observar as métricas de classificação.


In [42]:

y_pred = modelo.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, zero_division=0))


Accuracy: 1.0

              precision    recall  f1-score   support

   devolucao       1.00      1.00      1.00         4
     estoque       1.00      1.00      1.00         3
      pedido       1.00      1.00      1.00         3
       preco       1.00      1.00      1.00         3
    saudacao       1.00      1.00      1.00         3

    accuracy                           1.00        16
   macro avg       1.00      1.00      1.00        16
weighted avg       1.00      1.00      1.00        16



In [43]:
y_train_pred = modelo.predict(X_train)
y_test_pred = modelo.predict(X_test)

print("Treinamento:")
print(accuracy_score(y_train, y_train_pred))

print("\nTeste:")
print(accuracy_score(y_test, y_test_pred))

print("\n\n")
erros = X_test[y_test != y_pred]
for texto, real, previsto in zip(X_test[y_test != y_pred], y_test[y_test != y_pred], y_pred[y_test != y_pred]):
    print(f"Frase: {texto} | Real: {real} | Previsto: {previsto}")

Treinamento:
1.0

Teste:
1.0






### Como interpretar?

- **Precision:** entre as mensagens que o modelo classificou como uma determinada intenção, quantas estavam corretas?
- **Recall:** entre as mensagens que realmente pertenciam a uma intenção, quantas o modelo conseguiu encontrar?
- **F1-score:** combina precision e recall em uma única medida.

### Debate

Um modelo com 100% de acerto neste pequeno dataset significa que temos um chatbot pronto para produção?

**Não.**

O conjunto é pequeno e controlado. Em uma aplicação real, seria necessário avaliar generalização, qualidade das respostas, situações inesperadas, dados novos e outros aspectos.



## Pré-processamento
Para esta primeira implementação em português, vamos começar com uma normalização simples, evitando dependências adicionais de recursos linguísticos.


In [44]:

def preprocessar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záàâãéêíóôõúç0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

exemplos = [
    "Olá! Quanto custa o Notebook?",
    "TEM CELULAR DISPONÍVEL???",
    "Quero acompanhar meu pedido #12345."
]

for texto in exemplos:
    print("Original:   ", texto)
    print("Processado: ", preprocessar(texto))
    print()


Original:    Olá! Quanto custa o Notebook?
Processado:  olá quanto custa o notebook

Original:    TEM CELULAR DISPONÍVEL???
Processado:  tem celular disponível

Original:    Quero acompanhar meu pedido #12345.
Processado:  quero acompanhar meu pedido 12345




## Extração de entidades
Nesta atividade vamos implementar uma extração simples de número de pedido com expressão regular.

Isso é uma **regra**, não um modelo avançado de reconhecimento de entidades.


In [45]:

def extrair_pedido(texto):
    resultado = re.search(r"(?:pedido|ordem|compra)\s*#?\s*(\d+)", texto.lower())
    return resultado.group(1) if resultado else None

for mensagem in [
    "Quero acompanhar o pedido 12345",
    "Qual o status da compra #9876?",
    "Quero rastrear meu pedido"
]:
    print(mensagem, "->", extrair_pedido(mensagem))


Quero acompanhar o pedido 12345 -> 12345
Qual o status da compra #9876? -> 9876
Quero rastrear meu pedido -> None


In [48]:
def extrair_produto(texto):
    marcas_conhecidas = [
        "dell", "samsung", "apple", "motorola", "lenovo",
        "asus", "lg", "xiaomi", "hp", "acer"
    ]
    texto_lower = texto.lower()
    for marca in marcas_conhecidas:
        if marca in texto_lower:
            return marca.capitalize()
    return None

for mensagem in [
    "Quero saber o preço do notebook Dell",
    "Tem celular Samsung disponível?",
    "Quanto custa o notebook?",
]:
    print(mensagem, "-> produto:", extrair_produto(mensagem))

Quero saber o preço do notebook Dell -> produto: Dell
Tem celular Samsung disponível? -> produto: Samsung
Quanto custa o notebook? -> produto: None



## Geração de linguagem natural (NLG)
O chatbot não vai gerar texto livre como um grande modelo de linguagem. Ele selecionará uma resposta adequada à intenção identificada.


In [50]:

respostas = {
    "saudacao": [
        "Olá! Como posso ajudar?",
        "Oi! Em que posso ajudar você?",
        "Olá! Posso ajudar com preços, estoque ou pedidos."
    ],
    "preco": [
        "Posso ajudar a consultar o preço do produto.",
        "Claro! Vou verificar o preço para você."
    ],
    "estoque": [
        "Vou verificar a disponibilidade do produto.",
        "Claro! Vou consultar o estoque."
    ],
    "pedido": [
        "Vou ajudar você a acompanhar o pedido.",
        "Claro! Vamos verificar o status da sua entrega."
    ],
    "devolucao": [
        "Vou te ajudar com a devolução do produto.",
        "Claro! Vamos iniciar o processo de devolução."
    ]
}

def gerar_resposta(intencao, mensagem):
    if intencao == "pedido":
        numero = extrair_pedido(mensagem)
        if numero:
            return f"Vou verificar o status do pedido #{numero}."

    return random.choice(respostas.get(
        intencao,
        ["Desculpe, não consegui entender sua solicitação."]
    ))



## Montando o chatbot

Agora vamos integrar:

**entrada → pré-processamento → NLU → intenção/entidade → NLG → resposta**


In [53]:

def chatbot(mensagem):
    texto = preprocessar(mensagem)
    intencao = modelo.predict([texto])[0]
    produto = extrair_produto(mensagem)
    resposta = gerar_resposta(intencao, mensagem)

    return {
        "mensagem": mensagem,
        "intencao": intencao,
        "produto": produto,
        "resposta": resposta
    }

testes = [
    "Oi, tudo bem?",
    "Quanto custa o notebook?",
    "Tem celular disponível?",
    "Quero acompanhar o pedido 12345",
    "Quero saber o preço do notebook Dell"
]

for mensagem in testes:
    resultado = chatbot(mensagem)
    print("Usuário:", resultado["mensagem"])
    print("\nIntenção:", resultado["intencao"])
    print("Produto:", resultado["produto"])
    print("\nResposta:", resultado["resposta"])
    print("-" * 60)


Usuário: Oi, tudo bem?

Intenção: saudacao
Produto: None

Resposta: Olá! Como posso ajudar?
------------------------------------------------------------
Usuário: Quanto custa o notebook?

Intenção: preco
Produto: None

Resposta: Claro! Vou verificar o preço para você.
------------------------------------------------------------
Usuário: Tem celular disponível?

Intenção: estoque
Produto: None

Resposta: Claro! Vou consultar o estoque.
------------------------------------------------------------
Usuário: Quero acompanhar o pedido 12345

Intenção: pedido
Produto: None

Resposta: Vou verificar o status do pedido #12345.
------------------------------------------------------------
Usuário: Quero saber o preço do notebook Dell

Intenção: preco
Produto: Dell

Resposta: Posso ajudar a consultar o preço do produto.
------------------------------------------------------------
